# Bias Node Research Notebook

Generic notebook for testing bias nodes and generating comprehensive reports.

## How to Use:
1. **Edit the `BIAS_NODES_TO_TEST` list** in the configuration cell below
2. Add/remove any bias nodes you want to test
3. Run all cells to generate:
   - Feature analysis reports (decile plots, distributions, permutation tests)
   - QuantStats HTML tearsheets (Sharpe, Sortino, drawdowns, etc.)

## Output:
- Reports saved to: `research/reports/bias_nodes/{node_name}/`
- Tearsheets saved to: `research/reports/bias_nodes/{node_name}/{node_name}_tearsheet.html`

In [ ]:
import sys
from pathlib import Path

# Get project root
project_root = Path.cwd() if (Path.cwd() / 'vault').exists() else Path.cwd().parent
sys.path.insert(0, str(project_root))

# Create reports directory
reports_dir = project_root / 'research' / 'reports' / 'bias_nodes'
reports_dir.mkdir(parents=True, exist_ok=True)
print(f"Reports will be saved to: {reports_dir}")

In [ ]:
from utils.enums import Ticker, TimeFrame
from datetime import datetime
import pandas as pd
import numpy as np

from feature_extraction.feature_extractor import extract_features_for_bias_node
from feature_selection.base_models import BaseModel, QuantileBinningModel, TwoBinBinningModel
from eda.feature_explorer import FeatureExplorer
from metrics.plotting.graphing.quantstats_reports import generate_tearsheet
import utils.helpers as helpers

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

---
## Configuration

**Edit the `BIAS_NODES_TO_TEST` list below to add/remove bias nodes.**

Each entry is a dict with:
- `name`: Human-readable name for reports
- `spec`: Bias node specification (module_name, timeframes, params)
- `is_continuous`: True for continuous features (0-100), False for rule-based (0/1)

In [ ]:
# =============================================================================
# CONFIGURATION - Edit this cell to customize your research
# =============================================================================

# Tickers to test on (ensemble)
ENSEMBLE_TICKERS = [Ticker.ES, Ticker.NQ, Ticker.YM, Ticker.RTY]

# Date range for analysis
START_DATE = datetime(2000, 1, 1)
END_DATE = datetime(2024, 12, 31)

# Permutation test repetitions (higher = more accurate p-values but slower)
PERMUTATION_REPS = 100

# Generate QuantStats tearsheets (set to False to skip)
GENERATE_TEARSHEETS = True

# =============================================================================
# BIAS NODES TO TEST - Add/remove entries as needed
# =============================================================================
# Each entry needs:
#   - 'name': Display name for reports
#   - 'spec': {'module_name': ..., 'timeframes': [...], 'params': {...}}
#   - 'is_continuous': True for continuous (use QuantileBinning), False for rule-based (TwoBinBinning)

BIAS_NODES_TO_TEST = [
    # ---------------------------------------------------------------------
    # CONTINUOUS FEATURES (0-100 or similar range) - Use QuantileBinning
    # ---------------------------------------------------------------------
    {
        'name': 'Adaptive_RSI',
        'spec': {
            'module_name': 'adaptive_rsi',
            'timeframes': [TimeFrame.D],
            'params': {'min_period': [2, 3], 'max_period': [10, 14], 'volatility_period': [20]}
        },
        'is_continuous': True
    },
    {
        'name': 'Casey_C',
        'spec': {
            'module_name': 'casey_c',
            'timeframes': [TimeFrame.D],
            'params': {'change_period': [5, 10], 'ranking_period': [50, 100]}
        },
        'is_continuous': True
    },
    {
        'name': 'Cyclical_RSI',
        'spec': {
            'module_name': 'cyclical_rsi',
            'timeframes': [TimeFrame.D],
            'params': {'short_period': [5], 'long_period': [20], 'rsi_period': [14]}
        },
        'is_continuous': True
    },
    {
        'name': 'Detrended_RSI',
        'spec': {
            'module_name': 'detrended_rsi',
            'timeframes': [TimeFrame.D],
            'params': {'rsi_period': [14], 'detrend_period': [50, 100]}
        },
        'is_continuous': True
    },
    {
        'name': 'DeMark_REI',
        'spec': {
            'module_name': 'demark_rei',
            'timeframes': [TimeFrame.D],
            'params': {'period': [5, 8, 13]}
        },
        'is_continuous': True
    },
    {
        'name': 'RSI_Percentile',
        'spec': {
            'module_name': 'rsi_percentile',
            'timeframes': [TimeFrame.D],
            'params': {'rsi_period': [14], 'percentile_period': [126, 252]}
        },
        'is_continuous': True
    },
    {
        'name': 'Stochastic_RSI',
        'spec': {
            'module_name': 'stochastic_rsi',
            'timeframes': [TimeFrame.D],
            'params': {'rsi_period': [14], 'stoch_period': [14], 'smooth_k': [3]}
        },
        'is_continuous': True
    },
    {
        'name': 'TSI',
        'spec': {
            'module_name': 'tsi',
            'timeframes': [TimeFrame.D],
            'params': {'long_period': [25], 'short_period': [13]}
        },
        'is_continuous': True
    },
    {
        'name': 'ZScore_RSI',
        'spec': {
            'module_name': 'zscore_rsi',
            'timeframes': [TimeFrame.D],
            'params': {'rsi_period': [14], 'zscore_period': [50, 100]}
        },
        'is_continuous': True
    },
    
    # ---------------------------------------------------------------------
    # RULE-BASED FEATURES (0/1 or -1/0/1 signals) - Use TwoBinBinning
    # ---------------------------------------------------------------------
    {
        'name': 'Percent_B',
        'spec': {
            'module_name': 'percent_b',
            'timeframes': [TimeFrame.D],
            'params': {'period': [20], 'std_dev': [2.0], 'lower_threshold': [0.0, 0.2]}
        },
        'is_continuous': False
    },
    {
        'name': 'Double_7s',
        'spec': {
            'module_name': 'double7s',
            'timeframes': [TimeFrame.D],
            'params': {'short_period': [7], 'ma_period': [200]}
        },
        'is_continuous': False
    },
    {
        'name': 'RSI_Signal',
        'spec': {
            'module_name': 'rsi_signal',
            'timeframes': [TimeFrame.D],
            'params': {'rsi_period': [14], 'oversold': [30], 'overbought': [70], 'mode': ['threshold']}
        },
        'is_continuous': False
    },
    {
        'name': 'SuperTrend_Cross',
        'spec': {
            'module_name': 'supertrend_cross',
            'timeframes': [TimeFrame.D],
            'params': {'atr_period': [10], 'fast_multiplier': [2.0], 'slow_multiplier': [3.0]}
        },
        'is_continuous': False
    },
]

print(f"Configuration loaded:")
print(f"  Tickers: {[t.name for t in ENSEMBLE_TICKERS]}")
print(f"  Date range: {START_DATE.date()} to {END_DATE.date()}")
print(f"  Nodes to test: {len(BIAS_NODES_TO_TEST)}")
print(f"  Generate tearsheets: {GENERATE_TEARSHEETS}")

---
## Helper Functions

In [ ]:
def test_bias_node(bias_spec: dict, node_name: str, is_continuous: bool = True,
                   show_plots: bool = False, export: bool = True):
    """
    Test a bias node and generate summary report.
    
    Returns:
        results: Dictionary with analysis results, or None if failed
    """
    print(f"\n{'='*70}")
    print(f"Testing: {node_name}")
    print(f"{'='*70}")
    
    # Extract features
    print("\n[1/3] Extracting features...")
    try:
        features_df, targets_df = extract_features_for_bias_node(
            bias_spec=bias_spec,
            ticker=ENSEMBLE_TICKERS,
            start=START_DATE,
            end=END_DATE,
            use_millisecond_offset=True,
            target_col='log_return'
        )
        print(f"  [OK] Features shape: {features_df.shape}")
        print(f"  [OK] Targets shape: {targets_df.shape}")
        feature_cols = [col for col in features_df.columns if col != 'ticker']
        print(f"  [OK] Feature columns: {len(feature_cols)}")
    except Exception as e:
        print(f"  [FAIL] Feature extraction failed: {e}")
        return None
    
    # Build metadata
    print("\n[2/3] Building metadata...")
    metadata = {'feature_metadata': {}}
    for feature_col in features_df.columns:
        if feature_col != 'ticker':
            parsed = helpers.parse_feature_column_name(feature_col)
            metadata['feature_metadata'][feature_col] = {
                'module': parsed.get('module'),
                'parameters': parsed.get('params', {}),
                'timeframes': [parsed.get('tf')] if parsed.get('tf') else [],
                'base_name': parsed.get('module'),
                'full_name': feature_col
            }
    
    # Create FeatureExplorer
    explorer = FeatureExplorer(
        features_df=features_df,
        targets_df=targets_df,
        metadata=metadata
    )
    print(f"  [OK] FeatureExplorer created with {explorer.n_features} features, {explorer.n_samples} samples")
    
    # Select binning model
    if is_continuous:
        binning_model = QuantileBinningModel(n_bins=10, selection_metric='sortino', strategy='long')
    else:
        binning_model = TwoBinBinningModel(selection_metric='sortino', strategy='long')
    
    # Generate summary report
    print("\n[3/3] Generating summary report...")
    export_path = str(reports_dir / node_name.lower().replace(' ', '_').replace('%', 'pct'))
    
    try:
        results = explorer.generate_summary_report(
            binning_model=binning_model,
            target_col='log_return',
            strategy='long',
            show_plots=show_plots,
            verbose=False,
            export_report=export,
            export_path=export_path if export else None,
            permutation_test_nreps=PERMUTATION_REPS,
            include_ticker_plots=False
        )
        print(f"  [OK] Report generated successfully")
        if export:
            print(f"  [OK] Report exported to: {export_path}")
        
        # Print key metrics
        if results and 'permutation_results' in results and results['permutation_results'] is not None:
            perm_df = results['permutation_results']
            if len(perm_df) > 0:
                print(f"\n  Permutation Test Results:")
                for _, row in perm_df.iterrows():
                    sig = '[SIG]' if row['p_value'] <= 0.1 else '[---]'
                    print(f"    {sig} {row['feature'][:50]}: p={row['p_value']:.4f}")
        
        # Store features_df and targets_df in results for tearsheet generation
        results['features_df'] = features_df
        results['targets_df'] = targets_df
        results['binning_model'] = binning_model
        
        return results
        
    except Exception as e:
        print(f"  [FAIL] Report generation failed: {e}")
        import traceback
        traceback.print_exc()
        return None


def generate_node_tearsheet(bias_spec: dict, node_name: str, is_continuous: bool = True):
    """
    Generate a QuantStats HTML tearsheet for a bias node.
    Tests on ES ticker only for cleaner tearsheet.
    """
    from ensemble.portfolio_tester import calculate_strategy_returns_from_positions, calculate_log_returns_from_candles
    
    print(f"\n  Generating QuantStats tearsheet for {node_name}...")
    
    try:
        # Load candles for ES (single ticker for tearsheet)
        candles_df = helpers.load_data(
            ticker=Ticker.ES,
            timeframe=TimeFrame.D,
            start=START_DATE,
            end=END_DATE
        )
        candles_df['ticker'] = Ticker.ES
        candles_df['timeframe'] = TimeFrame.D
        
        # Create and fit BaseModel
        if is_continuous:
            binning_model = QuantileBinningModel(n_bins=10, selection_metric='sortino', strategy='long')
        else:
            binning_model = TwoBinBinningModel(selection_metric='sortino', strategy='long')
        
        base_model = BaseModel(
            feature_config={'bias_node_spec': bias_spec},
            binning_model=binning_model,
            tickers=[Ticker.ES]
        )
        
        # Compute targets
        target_series = pd.Series(
            np.log(candles_df['close'] / candles_df['open']).values,
            index=pd.DatetimeIndex(candles_df['datetime']),
            name='log_return'
        )
        
        # Fit the model
        base_model.fit(candles_df, target_series)
        
        # Generate predictions
        predictions = base_model.predict(candles_df, strategy='long')
        
        # Create positions DataFrame
        positions_df = pd.DataFrame({
            'datetime': predictions.index,
            'ticker': 'ES',
            'position_fraction': predictions.values.astype(float)
        }).set_index('datetime')
        
        # Calculate strategy returns
        strategy_returns = calculate_strategy_returns_from_positions(
            positions_df, candles_df, strategy='long'
        )
        
        # Calculate baseline (buy & hold) returns
        baseline_returns = calculate_log_returns_from_candles(candles_df)
        
        # Generate tearsheet
        node_dir = reports_dir / node_name.lower().replace(' ', '_').replace('%', 'pct')
        node_dir.mkdir(parents=True, exist_ok=True)
        tearsheet_path = str(node_dir / f"{node_name.lower().replace(' ', '_')}_tearsheet.html")
        
        generate_tearsheet(
            strategy_returns=strategy_returns,
            baseline_returns=baseline_returns,
            feature_name=f"{node_name} vs Buy & Hold (ES)",
            output_file=tearsheet_path,
            mode='html'
        )
        
        print(f"  [OK] Tearsheet saved to: {tearsheet_path}")
        return tearsheet_path
        
    except Exception as e:
        print(f"  [FAIL] Tearsheet generation failed: {e}")
        return None

---
## Run All Tests

This cell iterates through all bias nodes in `BIAS_NODES_TO_TEST` and generates reports.

In [ ]:
# Run tests for all bias nodes
all_results = {}
tearsheet_paths = {}

print("="*70)
print(f"STARTING BIAS NODE RESEARCH - {len(BIAS_NODES_TO_TEST)} nodes to test")
print("="*70)

for i, node_config in enumerate(BIAS_NODES_TO_TEST, 1):
    node_name = node_config['name']
    bias_spec = node_config['spec']
    is_continuous = node_config['is_continuous']
    
    print(f"\n[{i}/{len(BIAS_NODES_TO_TEST)}] Processing {node_name}...")
    
    # Generate feature analysis report
    results = test_bias_node(
        bias_spec=bias_spec,
        node_name=node_name,
        is_continuous=is_continuous,
        show_plots=False,
        export=True
    )
    all_results[node_name] = results
    
    # Generate QuantStats tearsheet if enabled
    if GENERATE_TEARSHEETS and results is not None:
        tearsheet_path = generate_node_tearsheet(
            bias_spec=bias_spec,
            node_name=node_name,
            is_continuous=is_continuous
        )
        tearsheet_paths[node_name] = tearsheet_path

print("\n" + "="*70)
print("ALL TESTS COMPLETE")
print("="*70)

---
## Summary of Results

In [ ]:
# Compile summary of all results
print("\n" + "="*70)
print("SUMMARY: Bias Node Research Results")
print("="*70)

summary_data = []
for name, result in all_results.items():
    if result is not None and 'permutation_results' in result and result['permutation_results'] is not None:
        perm_df = result['permutation_results']
        if len(perm_df) > 0:
            # Get best feature by p-value
            best_row = perm_df.loc[perm_df['p_value'].idxmin()]
            summary_data.append({
                'Node': name,
                'Best Feature': best_row['feature'][:40] + '...' if len(best_row['feature']) > 40 else best_row['feature'],
                'P-Value': best_row['p_value'],
                'Significant': 'YES' if best_row['p_value'] <= 0.1 else 'NO',
                'Status': 'OK'
            })
        else:
            summary_data.append({'Node': name, 'Status': 'No permutation results'})
    else:
        summary_data.append({'Node': name, 'Status': 'FAILED' if result is None else 'No results'})

summary_df = pd.DataFrame(summary_data)
print("\n")
print(summary_df.to_string(index=False))

# Count significant features
if 'Significant' in summary_df.columns:
    n_significant = (summary_df['Significant'] == 'YES').sum()
    n_total = len(summary_df[summary_df['Status'] == 'OK'])
    print(f"\n{n_significant}/{n_total} nodes have statistically significant features (p <= 0.1)")

print(f"\nAll reports saved to: {reports_dir}")

In [ ]:
# List all generated report files
import os

print("\nGenerated Report Files:")
print("="*70)

for root, dirs, files in os.walk(reports_dir):
    level = root.replace(str(reports_dir), '').count(os.sep)
    indent = ' ' * 2 * level
    folder_name = os.path.basename(root)
    if folder_name:
        print(f"{indent}[DIR] {folder_name}/")
    for file in files:
        print(f"{indent}  - {file}")

---
## QuantStats Tearsheet Links

Open these HTML files in your browser to view detailed performance metrics.

In [ ]:
if GENERATE_TEARSHEETS and tearsheet_paths:
    print("\nQuantStats Tearsheets Generated:")
    print("="*70)
    for node_name, path in tearsheet_paths.items():
        if path:
            print(f"  {node_name}: {path}")
else:
    print("No tearsheets generated (GENERATE_TEARSHEETS=False or all failed)")